# Pretraining notebook

## Section 1: Generate text before training

Load your current model

Enter a few prompts and note the output.

Try these:
 - love is
 - baby
 - tonight

Generate 20 tokens and note the output.

Reflection: Is there any quality or meaning to the output or is it largely random text?  Why?

In [2]:
import torch
import tiktoken
import sys
sys.path.append("../src")

# This makes results more reproducible and reduces confusion when comparing outputs.
torch.manual_seed(123)

from weird_ai.model import WeirdAIModel

tokenizer = tiktoken.get_encoding("gpt2")

def generate(model, idx, max_new_tokens, context_size,
             temperature=0.0, top_k=None, eos_id=None):
    for _ in range(max_new_tokens):            #1
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]
        if top_k is not None:                #2
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(
                logits < min_val, torch.tensor(float('-inf')).to(logits.device),
                logits
            )
        if temperature > 0.0:                  #3
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
        else:    #4
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)
        if idx_next == eos_id:              #5
            break
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print(device)

GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 256,    #1
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12, 
    "drop_rate_emb": 0.1,      
    "qkv_bias": False
}
# TODO:
# Load your model

model = WeirdAIModel(GPT_CONFIG_124M)
model.to(device)

# TODO:
# Generate text from:
# "love is"
# "baby"
# "tonight"
for prompt in ["love is", "baby", "tonight"]:
    idx = torch.tensor(tokenizer.encode(prompt)).unsqueeze(0).to(device)
    out = generate(
        model,
        idx,
        max_new_tokens=20,
        context_size=GPT_CONFIG_124M["context_length"],
    )
    print(tokenizer.decode(out.squeeze(0).tolist()))

cuda
love is cruiser experimented accumulation hashing Konami colonBIT Wein Hollandework teeth Lisp Somew nutritious87esters Coch 1080 slated consultants
baby Peergov battlegroundearth UCH agreementthel Slowly Irving nationality Systems attainment humiliatingarser seq imprint Flavoring redress millionsSharp
tonight antibioticsHD Clayton ratio publishers sv Leafs failed Clayton LINK supporters oppSummerJoshQuestEW Corpsanco Olparen


## Section 2: Understanding Logits

Using the logits starter code, do the following:
 - Apply softmax
 - Identify the highest probability token

Reflection: Why is softmax necessary before interpreting logits as probabilities?

In [3]:
logits = torch.tensor([
    [1.5, 2.0, 0.5]
])

probabilities = torch.softmax(logits, dim=-1)

print(probabilities)
print(probabilities.sum())

predicted_token = torch.argmax(probabilities)

print(predicted_token)

tensor([[0.3315, 0.5465, 0.1220]])
tensor(1.0000)
tensor(1)


## Section 3: Cross Entropy Loss

### Textbook 
The textbook discussion begins on page 136

### Manually calculate
 - Probabilities
 - Log probabilities
 - Average negative log probabilities

### Compare with torch
Compare your manual results to the torch cross entropy results

```python
     torch.nn.functional.cross_entropy(...)
'''


In [4]:
import torch

# Manually calculate values

probs = torch.tensor([
    0.7,
    0.2,
    0.1
])

target_index = 0

In [5]:
target_probability = probs[target_index]

print(target_probability)

tensor(0.7000)


In [6]:
log_probability = torch.log(target_probability)

print(log_probability)

tensor(-0.3567)


In [7]:
loss = -log_probability

print(loss)

tensor(0.3567)


In [8]:
# Compare with torch calculations
logits = torch.log(probs).unsqueeze(0)  
target = torch.tensor([target_index])    

torch_loss = torch.nn.functional.cross_entropy(logits, target)
print(torch_loss)  

tensor(0.3567)


## Section 4: Perplexity

Compute the perplexity

```python
perplexity = torch.exp(loss)
```

**Note:** A perplexity of 10 means the model is roughly as uncertain as choosing among 10 equally likely next tokens.

In [9]:
loss = torch.tensor(2.5)

perplexity = torch.exp(loss)

print(perplexity)

tensor(12.1825)


## Section 5: Understanding the Training Loop

The training loop performs:

1. Iterate through epochs
2. Iterate through batches
3. Zero gradients
4. Calculate loss
5. Backpropagation
6. Optimizer step
7. Evaluate model
8. Generate sample text

Draw or explain this process in your own words.

One epoch is a pass through all of our training data. We split each pass into smaller batches since it would be far too much data to process at once. Zeroing the gradients adds our new gradients to old ones so we clear them before each batch.
Loss is where we run the model and cross entropy between predictions and targets. Backpropagation helps us to adjust weights in a way that pushes us towards smaller loss. The optimizer changes the weights in the proper direction so that the model can actually learn.
We evaluate the model so that we can measure training and validation loss, and make sure we are training on patterns and not just exact repetition. We generate sample text so we can as humans understand how the output is changing.

# Training vs. Validation

You will:
 - Split lyrics into train/validation
 - Create loaders
 - Compute initial loss

Keep note of your: 
 - Training Loss: 
 - Validation Loss: 

In [10]:
from weird_ai.dataset import LyricsDataset

train_ratio = 0.9

from weird_ai.config import SAMPLE_LYRICS_FILE

with open(SAMPLE_LYRICS_FILE, "r", encoding="utf-8") as f:
    text_data = f.read()
# TODO:
# Split the corpus into train and validation text

split_idx = int(train_ratio * len(text_data))

train_text = text_data[:split_idx]
val_text = text_data[split_idx:]

print("Training characters:", len(train_text))
print("Validation characters:", len(val_text))

Training characters: 12156964
Validation characters: 1350774


In [11]:
from torch.utils.data import Dataset, DataLoader

# Create datasets
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt)    #1

        for i in range(0, len(token_ids) - max_length, stride):     #2
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):    #3
        return len(self.input_ids)

    def __getitem__(self, idx):         #4
        return self.input_ids[idx], self.target_ids[idx]

# Create dataloaders
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                        stride=128, shuffle=True, drop_last=True,
                        num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

train_loader = create_dataloader_v1(
        train_text,
        batch_size=2,
        max_length=GPT_CONFIG_124M["context_length"],
        stride=GPT_CONFIG_124M["context_length"],
        drop_last=True,
        shuffle=True,
        num_workers=0
    )
val_loader = create_dataloader_v1(
        val_text,
        batch_size=2,
        max_length=GPT_CONFIG_124M["context_length"],
        stride=GPT_CONFIG_124M["context_length"],
        drop_last=False,
        shuffle=False,
        num_workers=0
    )
print(train_loader)
print(val_loader)

In [12]:
# TODO:
# Calculate initial losses
from weird_ai.losses import calc_loss_loader

with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=10)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=10)

print("Training loss:", train_loss)
print("Validation loss:", val_loss)

Training loss: 10.990934276580811
Validation loss: 11.002517604827881


## Section 6: Training

Train for one Epoch
```python
train_model(...)
```

Then record the loss before and after training.  

In [ ]:
from weird_ai.trainer import train_model_simple, save_checkpoint
from weird_ai.losses import calc_loss_loader

num_epochs = 1

# TODO:
# Train the model
with torch.no_grad():
    initial_loss = calc_loss_loader(train_loader, model, device, num_batches=10)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)

train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=50, eval_iter=5,
    start_context="love is", tokenizer=tokenizer,
    context_size=GPT_CONFIG_124M["context_length"],
)

with torch.no_grad():
    final_loss = calc_loss_loader(train_loader, model, device, num_batches=10)

save_checkpoint(model, optimizer, num_epochs, train_losses, val_losses,
                tokens_seen, "../models/lesson-05-pretrained/checkpoint.pt")

Ep 1 Step 0: Train Loss = 9.7937, Val Loss = 9.8192, Tokens Seen = 512
Ep 1 Step 50: Train Loss = 6.6177, Val Loss = 6.8538, Tokens Seen = 26112
Ep 1 Step 100: Train Loss = 5.6395, Val Loss = 6.4616, Tokens Seen = 51712
Ep 1 Step 150: Train Loss = 6.1272, Val Loss = 6.3277, Tokens Seen = 77312
Ep 1 Step 200: Train Loss = 5.4170, Val Loss = 6.2526, Tokens Seen = 102912
Ep 1 Step 250: Train Loss = 5.1002, Val Loss = 6.1882, Tokens Seen = 128512
Ep 1 Step 300: Train Loss = 5.4775, Val Loss = 6.0903, Tokens Seen = 154112
Ep 1 Step 350: Train Loss = 5.2301, Val Loss = 6.0306, Tokens Seen = 179712
Ep 1 Step 400: Train Loss = 5.1249, Val Loss = 6.0067, Tokens Seen = 205312
Ep 1 Step 450: Train Loss = 5.1918, Val Loss = 5.9542, Tokens Seen = 230912
Ep 1 Step 500: Train Loss = 5.4586, Val Loss = 5.9677, Tokens Seen = 256512
Ep 1 Step 550: Train Loss = 5.0349, Val Loss = 5.8947, Tokens Seen = 282112
Ep 1 Step 600: Train Loss = 5.1499, Val Loss = 5.8472, Tokens Seen = 307712
Ep 1 Step 650: Train 

In [14]:
print(f"Initial loss: {initial_loss}")
print(f"Final loss: {final_loss}")

Initial loss: 10.99534206390381
Final loss: 4.372860431671143


In [3]:
from weird_ai.model import WeirdAIModel
from weird_ai.trainer import load_checkpoint
from weird_ai.generation import generate_and_print_sample

restored_model = WeirdAIModel(GPT_CONFIG_124M).to(device)
restored_optimizer = torch.optim.AdamW(restored_model.parameters(), lr=0.0004, weight_decay=0.1)

meta = load_checkpoint(restored_model, restored_optimizer,
                       "../models/lesson-05-pretrained/checkpoint.pt", device)

print("Epoch:", meta["epoch"])
print("Final train loss:", meta["train_losses"][-1])
print("Final val loss:", meta["val_losses"][-1])
print("Tokens seen:", meta["track_tokens_seen"][-1])

restored_model.eval()
for prompt in ["love is", "baby", "tonight"]:
    generate_and_print_sample(restored_model, tokenizer, device, prompt,
                              GPT_CONFIG_124M["context_length"])

Epoch: 1
Final train loss: 4.582479572296142
Final val loss: 5.299645137786865
Tokens seen: 3738112
love is the way [Chorus] I'm not the way [Chorus] I'ma-o-o-o-o-o-o-o-o-o-o-o-o-o-ha
baby [Chorus]   [Verse 1: Jung] I'ma] I'ma] I'm not ayy, I'ma gettin'Cause I'ma gettin'Cause I'ma get
tonight [Verse 1] [Verse 1: Jung] I'ma gettin'Cause I'ma gettin'Cause I'ma gettin'Cause I'ma] I'ma gettin'Cause I'm


## Evaluation

## Did the loss decrease?
It decreased considerably at first, and then slowly lowered loss as it continued to train.
## Why is decreasing loss important?
Loss is a measure of how well the model would have properly guessed the tokens from the validation set. A lower number means there was a higher chance of the model getting the correct tokens.
A loss of 11 means the next token is random from a subset of 50000 tokens. At a loss of 4.37 it seems to be right about 1/79 times. That doesnt seem great, but it is a massive leap.

## Before and After Comparison

Prompt:

love is

Before Training: love is cruiser experimented accumulation hashing Konami colonBIT Wein Hollandework teeth Lisp Somew nutritious87esters Coch 1080 slated consultants
__________________

After Training: love is the way [Chorus] I'm not the way [Chorus] I'ma-o-o-o-o-o-o-o-o-o-o-o-o-o-ha
__________________

Reflection:

How did the output change? Well it got much more intelligible and has learned to use the [Chorus] tag

What evidence do you see that the model learned something from the training data? The output genuinely looks like it could be song lyrics.